# Item-based collaborative filtering (Steam)

Builds an item-item cosine similarity matrix from `user_history.tsv` (the output of `preprocess_steam.ipynb`), for use as a candidate/similarity lookup at serving time.

Restructured from the original `ItemCF.ipynb`: that notebook defined two implementations — a `dict`-of-`dict` version (`itemCFTrain` / `ItemMatrix_fn` / `ItemSimilarityMatrix_fn`) and a vectorized `numpy` version (`ItemMatrix_fn2` / `ItemSimilarityMatrix_fn2`) — but only ever ran and saved output from the vectorized one. This version keeps only that path.

# 0. Import & logging

In [ ]:
import os
import itertools
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
from local_package.config.data import STEAM_PROCESSED_DIR
from local_package.config.log import setup_logging, STEAM_PROCESS_LOG_DIR

In [ ]:
logger = setup_logging(name="item_cf", level=logging.INFO, to_file=True, log_dir=STEAM_PROCESS_LOG_DIR)

# 1. Configuration

In [ ]:
# Path
DATA_DIR = STEAM_PROCESSED_DIR / "chatbot"  # matches OUTPUT_DIR from preprocess_steam.ipynb

USER_HISTORY_FILE = DATA_DIR / "user_history.tsv"
ITEM_SIM_FILE = DATA_DIR / "item_sim.npy"

In [ ]:
SIMILARITY_DTYPE = np.float16  # halves the matrix's on-disk size; plenty of precision for top-k similarity lookups

# 2. Load user history

In [ ]:
def load_user_history(path):
    logger.info("Loading user history from %s", path)
    df = pd.read_csv(path)
    logger.info("Shape of user history: %s", df.shape)
    return df

In [ ]:
train_data = load_user_history(USER_HISTORY_FILE)

# 3. Build per-user item lists

In [ ]:
def build_user_item_dict(df, user_col="user_id", item_col="item_id"):
    logger.info("Grouping interactions into per-user item lists")
    return df.groupby(user_col)[item_col].apply(list).to_dict()

In [ ]:
user_item_dict = build_user_item_dict(train_data)

# 4. Item co-occurrence matrix

In [ ]:
def build_cooccurrence_matrix(n_items, user_item_dict):
    """Count, for every pair of items, how many users interacted with both.

    Symmetric by construction (i, j) and (j, i) are both incremented, which is what the
    cosine-similarity step in the next section assumes.
    """
    matrix = np.zeros((n_items, n_items))
    skipped_users = 0
    for user, items in tqdm(user_item_dict.items()):
        if len(items) <= 1:
            skipped_users += 1
            continue
        pairs = list(itertools.combinations(items, 2))
        x1, x2 = zip(*pairs)
        matrix[x1, x2] += 1
        matrix[x2, x1] += 1
    if skipped_users:
        logger.info("Skipped %d users with a single interaction (no pairs to count)", skipped_users)
    return matrix

In [ ]:
n_items = train_data["item_id"].max() + 1
item_matrix = build_cooccurrence_matrix(n_items, user_item_dict)
logger.info("Co-occurrence matrix shape: %s", item_matrix.shape)

In [ ]:
# sanity check: the matrix must be symmetric, so row sums and column sums should match exactly
asymmetry = (item_matrix.sum(0) - item_matrix.sum(1)).sum()
logger.info("Matrix symmetry check (expect 0.0): %s", asymmetry)

# 5. Item similarity matrix

In [ ]:
def cosine_similarity_matrix(matrix, eps=1e-10):
    """sim[i, j] = c_ij / sqrt(N_i * N_j), the standard ItemCF cosine similarity."""
    counts = matrix.sum(axis=0)
    norm = np.sqrt(np.outer(counts, counts)) + eps
    return matrix / norm

In [ ]:
item_sim = cosine_similarity_matrix(item_matrix).astype(SIMILARITY_DTYPE)
logger.info("Item similarity matrix dtype: %s, shape: %s", item_sim.dtype, item_sim.shape)

# 6. Save

In [ ]:
np.save(ITEM_SIM_FILE, item_sim)
logger.info("Saved item similarity matrix to %s", ITEM_SIM_FILE)